# Music Annotation Project V7.8: Independent-Data Builder with Sealed Outcomes

**Runtime:** CPU. Select **Runtime > Run all** after preparing the independent input folders.

This notebook builds the independent score-level dataset required by V7.8 while preventing validation outcomes from being inspected or used for tuning before the predictor dataset and source-group roster are locked.

## Input layout

```text
MusicAnnotationProject/independent_structural_validation_v7_8_builder/
  clean/                 # completely new clean pages
  corrected_masks/       # controlled masks, same page stems and dimensions
  score_manifest.csv     # page, source_group and provenance declarations
```

Required manifest columns:

```text
page,source_group,score_title,edition_or_source,acquisition_notes,excluded_from_development
```

## Output phases

1. **Build:** runs frozen `geometry_t80`, emits predictor-only records and a candidate lock manifest.
2. **Lock:** writes an immutable dataset lock containing hashes of inputs, predictor data, settings and source-group metadata.
3. **Seal truth:** after a valid lock exists, calculates clean-truth diagnostics and structural labels into a separate sealed file. It prints no outcome counts or examples.
4. **Release:** only when `RELEASE_LABELLED_DATASET=True` and the exact lock ID is supplied, merges the predictors and sealed labels into `component_dataset.csv` for V7.8.

Do not browse the sealed file before the validation policy is registered. The technical lock prevents accidental pre-lock leakage; it cannot prevent a person with Drive access from deliberately opening files.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless scikit-image pandas numpy


In [ ]:
import cv2,json,hashlib,math,os
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime,timezone

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
INPUT_ROOT=ROOT/'independent_structural_validation_v7_8_builder'
CLEAN_DIR=INPUT_ROOT/'clean';MASK_DIR=INPUT_ROOT/'corrected_masks';MANIFEST_PATH=INPUT_ROOT/'score_manifest.csv'
OUTPUT_ROOT=ROOT/'independent_structural_validation_v7_8'
BUILD_DIR=OUTPUT_ROOT/'builder';SEALED_DIR=OUTPUT_ROOT/'sealed';LOCKS_DIR=OUTPUT_ROOT/'locks'
for d in [CLEAN_DIR,MASK_DIR,OUTPUT_ROOT,BUILD_DIR,SEALED_DIR,LOCKS_DIR]:d.mkdir(parents=True,exist_ok=True)

# Controls. First run with CREATE_LOCK=False and RELEASE_LABELLED_DATASET=False.
CREATE_LOCK=False
EXPECTED_LOCK_ID=None
BUILD_SEALED_OUTCOMES=False
RELEASE_LABELLED_DATASET=False

# Frozen geometry_t80 and structural-label settings.
GEOMETRY_THRESHOLD=80
CONTEXT_MARGIN=24
MIN_COMPONENT_AREA=20
DARK_THRESHOLD=190
EDGE_LOW=70;EDGE_HIGH=170
HORIZONTAL_KERNEL=35
STRUCTURAL_THRESHOLDS={'ink_recall_min':0.80,'ink_f1_min':0.80,'edge_f1_min':0.75}
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def md5_file(p):
 h=hashlib.md5()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def sha256_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def object_hash(v):return hashlib.sha256(json.dumps(v,sort_keys=True,default=str).encode()).hexdigest()
def image_map(folder):return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED}
def atomic_json(path,value):
 temp=Path(str(path)+'.tmp');temp.write_text(json.dumps(value,indent=2,default=str));temp.replace(path)
def immutable(observed,candidate,mask):
 out=observed.copy();out[mask>0]=candidate[mask>0]
 assert np.array_equal(out[mask==0],observed[mask==0])
 return out

SETTINGS={'schema':'v7.8-builder-1','geometry_threshold':GEOMETRY_THRESHOLD,'context_margin':CONTEXT_MARGIN,'minimum_component_area':MIN_COMPONENT_AREA,'dark_threshold':DARK_THRESHOLD,'edge_thresholds':[EDGE_LOW,EDGE_HIGH],'horizontal_kernel':HORIZONTAL_KERNEL,'structural_thresholds':STRUCTURAL_THRESHOLDS}
SETTINGS_HASH=object_hash(SETTINGS)
print('Input:',INPUT_ROOT);print('Output:',OUTPUT_ROOT);print('Frozen settings hash:',SETTINGS_HASH)


## Validate completely new score-level inputs


In [ ]:
if not MANIFEST_PATH.exists():raise RuntimeError(f'Missing {MANIFEST_PATH}')
manifest=pd.read_csv(MANIFEST_PATH)
required={'page','source_group','score_title','edition_or_source','acquisition_notes','excluded_from_development'}
if required-set(manifest.columns):raise RuntimeError(f'Manifest missing: {sorted(required-set(manifest.columns))}')
if manifest.page.duplicated().any():raise RuntimeError('Manifest page values must be unique.')
flag=manifest.excluded_from_development.astype(str).str.lower().isin(['true','1','yes'])
if not flag.all():raise RuntimeError('Every score/page must explicitly declare excluded_from_development=True.')
clean_map=image_map(CLEAN_DIR);mask_map=image_map(MASK_DIR);pages=sorted(set(clean_map)&set(mask_map))
if set(manifest.page.astype(str))!=set(pages):
 raise RuntimeError(f'Manifest/files mismatch. Manifest only={sorted(set(manifest.page.astype(str))-set(pages))}; files only={sorted(set(pages)-set(manifest.page.astype(str)))}')
issues=[];input_records=[]
for page in pages:
 clean=cv2.imread(str(clean_map[page]),0);mask=cv2.imread(str(mask_map[page]),0)
 if clean is None or mask is None:issues.append((page,'unreadable'));continue
 if clean.shape!=mask.shape:issues.append((page,'shape mismatch'));continue
 input_records.append({'page':page,'clean_file':str(clean_map[page]),'mask_file':str(mask_map[page]),'clean_sha256':sha256_file(clean_map[page]),'mask_sha256':sha256_file(mask_map[page]),'height':clean.shape[0],'width':clean.shape[1]})
if issues:raise RuntimeError(f'Input problems: {issues}')
if manifest.source_group.nunique()<2:raise RuntimeError('At least two source groups are required to build; V7.8 validation requires broader coverage.')
print('Validated pages:',len(pages),'source groups:',manifest.source_group.nunique())


## Frozen geometry_t80, routing and predictor functions


In [ ]:
def paper_value(observed,mask):
 ring=cv2.dilate((mask>0).astype(np.uint8),np.ones((9,9),np.uint8),iterations=2).astype(bool)&~(mask>0)
 values=observed[ring]
 return int(np.percentile(values,75)) if values.size else 255

def geometry_support(observed,mask):
 visible=observed.copy();visible[mask>0]=255
 dark=(visible<DARK_THRESHOLD).astype(np.uint8)*255
 horizontal=cv2.morphologyEx(dark,cv2.MORPH_OPEN,np.ones((1,HORIZONTAL_KERNEL),np.uint8))>0
 vertical=cv2.morphologyEx(dark,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0
 compact=(dark>0)&~cv2.dilate(horizontal.astype(np.uint8),np.ones((3,5),np.uint8)).astype(bool)&~cv2.dilate(vertical.astype(np.uint8),np.ones((5,3),np.uint8)).astype(bool)
 support=cv2.dilate((horizontal|vertical|compact).astype(np.uint8),np.ones((3,3),np.uint8)).astype(bool)
 return horizontal,vertical,compact,support

def geometry_t80(clean,mask):
 # Controlled validation simulation: create an observed page without exposing clean truth to predictor functions.
 observed=clean.copy();observed[mask>0]=50
 horizontal,vertical,compact,support=geometry_support(observed,mask)
 m=mask>0;paper=paper_value(observed,mask)
 preserve=m&support
 remove=m&~support&(observed>=GEOMETRY_THRESHOLD)
 ambiguous=m&~preserve&~remove
 candidate=observed.copy();candidate[remove]=paper
 # Frozen conservative repair: nearest-paper estimate for ambiguous pixels; preserve pixels are byte-exact.
 candidate[ambiguous]=paper
 candidate[preserve]=observed[preserve]
 corrected=immutable(observed,candidate,mask)
 routing={'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact,'observed':observed}
 return corrected,routing

def bbox_context(component,shape,margin=CONTEXT_MARGIN):
 ys,xs=np.where(component);x0,x1=xs.min(),xs.max()+1;y0,y1=ys.min(),ys.max()+1;h,w=shape
 return max(0,x0-margin),max(0,y0-margin),min(w,x1+margin),min(h,y1+margin),(x0,y0,x1,y1)

def component_predictors(page,source_group,component_id,component,routing):
 observed=routing['observed'];x0,y0,x1,y1,bbox=bbox_context(component,observed.shape);bx0,by0,bx1,by1=bbox
 context=np.zeros_like(component);context[y0:y1,x0:x1]=True;context&=~component
 vals=observed[context];edges=cv2.Canny(observed,EDGE_LOW,EDGE_HIGH)>0
 area=int(component.sum());w=bx1-bx0;h=by1-by0;dist=cv2.distanceTransform(component.astype(np.uint8),cv2.DIST_L2,5)
 def frac(key):return float(np.sum(routing[key]&component)/area)
 return {'source_group':source_group,'page':page,'component':component_id,'w':w,'h':h,'area':area,'aspect_ratio':w/max(h,1),'fill_ratio':area/max(w*h,1),'touches_page_border':bool(bx0==0 or by0==0 or bx1==observed.shape[1] or by1==observed.shape[0]),'context_pixels':int(context.sum()),'context_edge_density':float(edges[context].mean()) if context.any() else 0.0,'context_dark_density':float((vals<DARK_THRESHOLD).mean()) if vals.size else 0.0,'context_mean_intensity':float(vals.mean()) if vals.size else 255.0,'context_intensity_std':float(vals.std()) if vals.size else 0.0,'maximum_mask_depth':float(dist[component].max()),'horizontal_fraction':frac('horizontal'),'vertical_fraction':frac('vertical'),'compact_fraction':frac('compact'),'preserve_fraction':frac('preserve'),'remove_fraction':frac('remove'),'ambiguous_fraction':frac('ambiguous'),'reconstruction_dependency_fraction':frac('ambiguous'),'component_pixels':area}


## Build predictor-only dataset


In [ ]:
meta=manifest.set_index('page');predictor_rows=[];runtime_cache={}
for page in pages:
 clean=cv2.imread(str(clean_map[page]),0);mask=np.where(cv2.imread(str(mask_map[page]),0)>0,255,0).astype(np.uint8)
 corrected,routing=geometry_t80(clean,mask)
 n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
 for component_id in range(1,n):
  if stats[component_id,cv2.CC_STAT_AREA]<MIN_COMPONENT_AREA:continue
  comp=labels==component_id
  predictor_rows.append(component_predictors(page,str(meta.loc[page,'source_group']),component_id,comp,routing))
 runtime_cache[page]={'clean':clean,'mask':mask,'corrected':corrected,'labels':labels}
predictors=pd.DataFrame(predictor_rows).sort_values(['source_group','page','component']).reset_index(drop=True)
PREDICTOR_PATH=BUILD_DIR/'component_predictors_locked_candidate.csv';predictors.to_csv(PREDICTOR_PATH,index=False)
input_df=pd.DataFrame(input_records).merge(manifest,on='page',validate='one_to_one');INPUT_MANIFEST_PATH=BUILD_DIR/'input_manifest_locked_candidate.csv';input_df.to_csv(INPUT_MANIFEST_PATH,index=False)
print('Predictor rows built:',len(predictors));print('Candidate predictor file:',PREDICTOR_PATH)


## Create immutable lock


In [ ]:
lock_payload={'schema':'v7.8-independent-lock-1','settings':SETTINGS,'settings_hash':SETTINGS_HASH,'predictor_sha256':sha256_file(PREDICTOR_PATH),'input_manifest_sha256':sha256_file(INPUT_MANIFEST_PATH),'score_manifest_sha256':sha256_file(MANIFEST_PATH),'pages':len(pages),'source_groups':int(manifest.source_group.nunique()),'components':len(predictors),'input_hashes':sorted(input_records,key=lambda x:x['page'])}
LOCK_ID=object_hash(lock_payload);LOCK_PATH=LOCKS_DIR/f'{LOCK_ID}.json'
if CREATE_LOCK:
 if LOCK_PATH.exists():
  existing=json.loads(LOCK_PATH.read_text())
  if existing.get('lock_id')!=LOCK_ID:raise RuntimeError('Existing lock mismatch.')
 else:
  atomic_json(LOCK_PATH,{**lock_payload,'lock_id':LOCK_ID,'created_utc':datetime.now(timezone.utc).isoformat(),'status':'LOCKED_BEFORE_OUTCOME_CALCULATION'})
 print('DATASET LOCK CREATED:',LOCK_ID)
else:
 print('CANDIDATE LOCK ID:',LOCK_ID);print('Inspect only inputs and predictor schema. Set CREATE_LOCK=True only when the source roster and settings are final.')


## Sealed clean-truth labelling after lock


In [ ]:
def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def tolerant_scores(ref,pred,region,tol=1):
 ref=ref&region;pred=pred&region
 if not ref.any():return np.nan,np.nan,np.nan
 k=np.ones((2*tol+1,2*tol+1),np.uint8);rd=cv2.dilate(ref.astype(np.uint8),k)>0;pd=cv2.dilate(pred.astype(np.uint8),k)>0
 precision=float(np.sum(pred&rd)/max(pred.sum(),1));recall=float(np.sum(ref&pd)/ref.sum());f1=2*precision*recall/max(precision+recall,1e-12);return precision,recall,f1

def truth_row(page,component_id,component,clean,corrected):
 ip,ir,if1=tolerant_scores(printed_ink(clean),printed_ink(corrected),component,1)
 ep,er,ef1=tolerant_scores(cv2.Canny(clean,EDGE_LOW,EDGE_HIGH)>0,cv2.Canny(corrected,EDGE_LOW,EDGE_HIGH)>0,component,1)
 reasons=[]
 if np.isfinite(ir) and ir<STRUCTURAL_THRESHOLDS['ink_recall_min']:reasons.append('ink_recall')
 if np.isfinite(if1) and if1<STRUCTURAL_THRESHOLDS['ink_f1_min']:reasons.append('ink_f1')
 if np.isfinite(ef1) and ef1<STRUCTURAL_THRESHOLDS['edge_f1_min']:reasons.append('edge_f1')
 return {'page':page,'component':component_id,'structural_safe':not reasons,'structural_outcome':'structural_safe' if not reasons else 'structural_unsafe','structural_failure_reasons':'|'.join(reasons),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_precision':ep,'edge_recall':er,'edge_f1':ef1}

if BUILD_SEALED_OUTCOMES:
 if EXPECTED_LOCK_ID!=LOCK_ID or not LOCK_PATH.exists():raise RuntimeError('Sealed outcome build requires EXPECTED_LOCK_ID matching an existing immutable lock.')
 locked=json.loads(LOCK_PATH.read_text())
 if locked['predictor_sha256']!=sha256_file(PREDICTOR_PATH) or locked['input_manifest_sha256']!=sha256_file(INPUT_MANIFEST_PATH):raise RuntimeError('Locked files changed after lock creation.')
 truth=[]
 for page in pages:
  cache=runtime_cache[page]
  for component_id in predictors.loc[predictors.page==page,'component'].astype(int):
   truth.append(truth_row(page,component_id,cache['labels']==component_id,cache['clean'],cache['corrected']))
 truth_df=pd.DataFrame(truth).sort_values(['page','component'])
 SEALED_PATH=SEALED_DIR/f'{LOCK_ID}_structural_truth.csv';truth_df.to_csv(SEALED_PATH,index=False)
 seal={'lock_id':LOCK_ID,'truth_sha256':sha256_file(SEALED_PATH),'rows':len(truth_df),'created_utc':datetime.now(timezone.utc).isoformat(),'outcomes_not_printed':True};atomic_json(SEALED_DIR/f'{LOCK_ID}_seal.json',seal)
 print('SEALED OUTCOME FILE CREATED. Outcome counts and examples intentionally suppressed.');print('Lock ID:',LOCK_ID)
else:print('Sealed outcomes not built. Set BUILD_SEALED_OUTCOMES=True only after the immutable lock is registered.')


## Explicit release for the separate V7.8 validator


In [ ]:
if RELEASE_LABELLED_DATASET:
 if EXPECTED_LOCK_ID!=LOCK_ID:raise RuntimeError('Release requires the exact registered EXPECTED_LOCK_ID.')
 SEALED_PATH=SEALED_DIR/f'{LOCK_ID}_structural_truth.csv';SEAL_PATH=SEALED_DIR/f'{LOCK_ID}_seal.json'
 if not LOCK_PATH.exists() or not SEALED_PATH.exists() or not SEAL_PATH.exists():raise RuntimeError('Lock or sealed truth is missing.')
 seal=json.loads(SEAL_PATH.read_text())
 if seal['truth_sha256']!=sha256_file(SEALED_PATH):raise RuntimeError('Sealed truth hash mismatch.')
 labelled=predictors.merge(pd.read_csv(SEALED_PATH),on=['page','component'],validate='one_to_one')
 labelled.to_csv(OUTPUT_ROOT/'component_dataset.csv',index=False)
 final_manifest=manifest.copy();final_manifest.to_csv(OUTPUT_ROOT/'score_manifest.csv',index=False)
 release={'lock_id':LOCK_ID,'component_dataset_sha256':sha256_file(OUTPUT_ROOT/'component_dataset.csv'),'score_manifest_sha256':sha256_file(OUTPUT_ROOT/'score_manifest.csv'),'released_utc':datetime.now(timezone.utc).isoformat(),'settings_hash':SETTINGS_HASH,'components':len(labelled),'source_groups':int(labelled.source_group.nunique())};atomic_json(OUTPUT_ROOT/'dataset_release.json',release)
 print('LABELLED DATASET RELEASED FOR V7.8 VALIDATION');print('Lock ID:',LOCK_ID);print('component_dataset.csv:',OUTPUT_ROOT/'component_dataset.csv')
else:print('Labelled dataset remains unreleased.')
